In [1]:
import pandas as pd
import numpy as np
from statsbombpy import sb

pd.set_option('display.max_columns', None)

In [6]:
import warnings
warnings.filterwarnings('ignore')

In [8]:
# get all the shots into a pandas data frame

COMPETITION_ID = 43
SEASON_ID = 3        

matches = sb.matches(competition_id=COMPETITION_ID, season_id=SEASON_ID)

all_shots = []
for match_id in matches['match_id']:
    ev = sb.events(match_id=match_id)
    all_shots.append(ev[ev['type'] == 'Shot'])

shots = pd.concat(all_shots, ignore_index=True).copy()
print(shots.shape)

(1706, 95)


In [9]:
# creating target value, goal = 1, no goal = 0

shots['is_goal'] = (shots['shot_outcome'] == 'Goal').astype(int)
print(shots['is_goal'].value_counts()) # num goals total
print("Base goal rate:", shots['is_goal'].mean().round(3)) # rate of goals in dataset

is_goal
0    1523
1     183
Name: count, dtype: int64
Base goal rate: 0.107


In [10]:
# put the x and y column into two seperate columns
shots[['x', 'y']] = pd.DataFrame(shots['location'].tolist(), index=shots.index)
print("Missing coordinates:", shots[['x', 'y']].isna().any(axis=1).sum()) # see which shots are missing an x,y

Missing coordinates: 0


In [11]:
GOAL_X = 120        # attacking goal line
GOAL_Y = 40         # centre of the goal
GOAL_WIDTH = 8      # posts sit at y=36 and y=44

# c  = how far the shot is from the goal LINE (straight-ahead distance)
# dy = how far off-centre it is (lateral offset from the goal's middle)
c = GOAL_X - shots['x']
dy = shots['y'] - GOAL_Y

shots['distance'] = np.sqrt(c**2 + dy**2) # distance to goal using pythagorean theorem
shots['angle'] = np.arctan2(GOAL_WIDTH * c, c**2 + dy**2 - (GOAL_WIDTH / 2)**2) # angle of the shooter to goal, trust me it's right

In [12]:
# check rate of goals per distance
shots['dist_bucket'] = pd.cut(shots['distance'], bins=[0, 6, 12, 18, 24, 30, 100])
print("Goal rate by distance:")
print(shots.groupby('dist_bucket', observed=True)['is_goal'].agg(['mean', 'count']))

# check rate of goals per angle
shots['angle_deg'] = np.degrees(shots['angle'])
shots['angle_bucket'] = pd.cut(shots['angle_deg'], bins=[0, 10, 20, 30, 45, 90])
print("\nGoal rate by shot angle:")
print(shots.groupby('angle_bucket', observed=True)['is_goal'].agg(['mean', 'count']))

Goal rate by distance:
                 mean  count
dist_bucket                 
(0, 6]       0.392857     56
(6, 12]      0.245146    412
(12, 18]     0.080645    372
(18, 24]     0.036585    328
(24, 30]     0.044248    339
(30, 100]    0.015075    199

Goal rate by shot angle:
                  mean  count
angle_bucket                 
(0, 10]       0.036364     55
(10, 20]      0.035323    821
(20, 30]      0.078488    344
(30, 45]      0.262458    301
(45, 90]      0.231214    173


In [13]:
# convert non numerical features into numbers for ML
categorical = pd.get_dummies(
    shots[['shot_body_part', 'shot_type']],
    prefix=['body', 'type']
).astype(int)

print(categorical.columns.tolist())
categorical.head()

['body_Head', 'body_Left Foot', 'body_Other', 'body_Right Foot', 'type_Free Kick', 'type_Open Play', 'type_Penalty']


,body_Head,body_Left Foot,body_Other,body_Right Foot,type_Free Kick,type_Open Play,type_Penalty
0,0,0,0,1,0,1,0
1,0,0,0,1,0,1,0
2,0,0,0,1,0,1,0
3,0,0,0,1,0,1,0
4,0,0,0,1,0,1,0


In [14]:
# penalties are different, they have a fixed rate of 0.75 for the most part
print(shots.groupby('shot_type', observed=True)['is_goal'].agg(['mean', 'count']))

               mean  count
shot_type                 
Free Kick  0.073171     82
Open Play  0.082905   1556
Penalty    0.705882     68


In [15]:
# assemble feature matrix X and target vector y
feature_cols = ['distance', 'angle']
X = pd.concat([shots[feature_cols], categorical], axis=1)
y = shots['is_goal']

print("X shape:", X.shape)
print("Features:", X.columns.tolist())
print("Any missing values:", X.isna().any().any())
X.head()

X shape: (1706, 9)
Features: ['distance', 'angle', 'body_Head', 'body_Left Foot', 'body_Other', 'body_Right Foot', 'type_Free Kick', 'type_Open Play', 'type_Penalty']
Any missing values: False


,distance,angle,body_Head,body_Left Foot,body_Other,body_Right Foot,type_Free Kick,type_Open Play,type_Penalty
0,31.016125,0.240497,0,0,0,1,0,1,0
1,5.099020,1.325818,0,0,0,1,0,1,0
2,16.643317,0.405354,0,0,0,1,0,1,0
3,30.083218,0.246831,0,0,0,1,0,1,0
4,12.529964,0.557962,0,0,0,1,0,1,0


In [16]:
# save dataset
import os
os.makedirs('../data', exist_ok=True)

dataset = X.copy()
dataset['is_goal'] = y.values
dataset.to_csv('../data/xg_dataset.csv', index=False)
print("Saved", dataset.shape, "to ml/data/xg_dataset.csv")

Saved (1706, 10) to ml/data/xg_dataset.csv
